# Data Preparation

Prepared by Afees Iyiola O. (UB 24047789)

This notebook prepares the IBM HR Analytics employee attrition dataset for the bias evaluation pipeline. It audits the raw data, removes columns that carry no usable information, defines the privileged and unprivileged groups for gender, age and marital status, encodes the features, and produces the train and test sets.

The split is created once here and written to disk. The model building, reweighing and equalised odds notebooks load that split rather than creating their own, so all three are measured on identical rows and remain comparable.

## Loading the dataset

Reads the raw CSV and checks the shape against the 1,470 records and 35 features reported in Chapter 3.2. The sample is drawn at random rather than taken from the top of the file, since the records may be ordered.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

RAW_PATH = Path("data/raw/ibm_dataset.csv")
RANDOM_SEED = 42

df = pd.read_csv(RAW_PATH)
print(df.shape)

df.sample(10, random_state=RANDOM_SEED)

(1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
1041,28,No,Travel_Rarely,866,Sales,5,3,Medical,1,1469,4,Male,84,3,2,Sales Executive,1,Single,8463,23490,0,Y,No,18,3,4,80,0,6,4,3,5,4,1,3
184,53,No,Travel_Rarely,1084,Research & Development,13,2,Medical,1,250,4,Female,57,4,2,Manufacturing Director,1,Divorced,4450,26250,1,Y,No,11,3,3,80,2,5,3,3,4,2,1,3
1222,24,Yes,Travel_Rarely,240,Human Resources,22,1,Human Resources,1,1714,4,Male,58,1,1,Human Resources,3,Married,1555,11585,1,Y,No,11,3,3,80,1,1,2,3,1,0,0,0
67,45,No,Travel_Rarely,1339,Research & Development,7,3,Life Sciences,1,86,2,Male,59,3,3,Research Scientist,1,Divorced,9724,18787,2,Y,No,17,3,3,80,1,25,2,3,1,0,0,0
220,36,No,Travel_Rarely,1396,Research & Development,5,2,Life Sciences,1,304,4,Male,62,3,2,Laboratory Technician,2,Single,5914,9945,8,Y,No,16,3,4,80,0,16,3,4,13,11,3,7
494,34,No,Travel_Rarely,204,Sales,14,3,Technical Degree,1,666,3,Female,31,3,1,Sales Representative,3,Divorced,2579,2912,1,Y,Yes,18,3,4,80,2,8,3,3,8,2,0,6
430,35,No,Travel_Rarely,144,Research & Development,22,3,Life Sciences,1,577,4,Male,46,1,1,Laboratory Technician,3,Single,4230,19225,0,Y,No,15,3,3,80,0,6,2,3,5,4,4,3
240,39,No,Travel_Rarely,1431,Research & Development,1,4,Medical,1,332,3,Female,96,3,1,Laboratory Technician,3,Divorced,2232,15417,7,Y,No,14,3,3,80,3,7,1,3,3,2,1,2
218,45,No,Non-Travel,1052,Sales,6,3,Medical,1,302,4,Female,57,2,3,Sales Executive,4,Single,8865,16840,6,Y,No,12,3,4,80,0,23,2,3,19,7,12,8
49,35,No,Travel_Rarely,1229,Research & Development,8,1,Life Sciences,1,63,4,Male,36,4,1,Laboratory Technician,4,Married,2269,4892,1,Y,No,19,3,4,80,0,1,2,3,1,0,0,1


## Structure and completeness

Checks the type of every column and whether any values are absent and datatypes.

In [2]:
df.info(verbose=True, show_counts=True)

missing = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(2),
})
print("\nTotal missing values:", int(missing["n_missing"].sum()))
print(missing[missing["n_missing"] > 0].to_string() or "")

# Missingness is sometimes written as a placeholder string, which isna() will not detect.
SENTINELS = {"unknown", "n/a", "na", "none", "null", "?", "-", "", "nan"}

encoded_missing = {}
for col in df.select_dtypes(include="object").columns:
    hits = sorted(set(df[col].astype(str).str.strip().str.lower().unique()) & SENTINELS)
    if hits:
        encoded_missing[col] = hits

print("Placeholder values found:", encoded_missing or "none")

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   str  
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EmployeeCount             1470 non-null   int64
 9   EmployeeNumber            1470 non-null   int64
 10  EnvironmentSatisfaction   1470 non-null   int64
 11  Gender                    1470 non-null   str  
 12  HourlyRate                1470 non-null   int64
 13  JobInvolvement            1470 non-null   int64
 14  JobLevel                  1470 non-null   int64
 15

C:\Users\iyiol\AppData\Local\Temp\ipykernel_12768\3422499183.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


## Value ranges and redundancy

Summary statistics, then checks for columns with no variance or a unique value per row.



In [4]:
display(df.describe().T)
display(df.describe(exclude="number").T)

cardinality = df.nunique().sort_values()
constant_cols = cardinality[cardinality == 1]
identifier_cols = cardinality[cardinality == len(df)]

print("Constant columns:")
print(constant_cols.to_string() if len(constant_cols) else "  none")

print("\nIdentifier-like columns:")
print(identifier_cols.to_string() if len(identifier_cols) else "  none")

print("\nExact duplicate rows:", df.duplicated().sum())

counts = df["Attrition"].value_counts()
percents = (df["Attrition"].value_counts(normalize=True) * 100).round(2)

print("\nTarget balance:")
print(pd.DataFrame({"count": counts, "percent": percents}).to_string())
print("Imbalance ratio: %.2f : 1" % (counts.max() / counts.min()))


,count,mean,std,min,25%,50%,75%,max
Age,1470.0,36.923810,9.135373,18.0,30.00,36.0,43.00,60.0
DailyRate,1470.0,802.485714,403.509100,102.0,465.00,802.0,1157.00,1499.0
DistanceFromHome,1470.0,9.192517,8.106864,1.0,2.00,7.0,14.00,29.0
Education,1470.0,2.912925,1.024165,1.0,2.00,3.0,4.00,5.0
EmployeeCount,1470.0,1.000000,0.000000,1.0,1.00,1.0,1.00,1.0
EmployeeNumber,1470.0,1024.865306,602.024335,1.0,491.25,1020.5,1555.75,2068.0
EnvironmentSatisfaction,1470.0,2.721769,1.093082,1.0,2.00,3.0,4.00,4.0
HourlyRate,1470.0,65.891156,20.329428,30.0,48.00,66.0,83.75,100.0
JobInvolvement,1470.0,2.729932,0.711561,1.0,2.00,3.0,3.00,4.0
JobLevel,1470.0,2.063946,1.106940,1.0,1.00,2.0,3.00,5.0


,count,unique,top,freq
Attrition,1470,2,No,1233
BusinessTravel,1470,3,Travel_Rarely,1043
Department,1470,3,Research & Development,961
EducationField,1470,6,Life Sciences,606
Gender,1470,2,Male,882
JobRole,1470,9,Sales Executive,326
MaritalStatus,1470,3,Married,673
Over18,1470,1,Y,1470
OverTime,1470,2,No,1054


Constant columns:
EmployeeCount    1
Over18           1
StandardHours    1

Identifier-like columns:
EmployeeNumber    1470

Exact duplicate rows: 0

Target balance:
           count  percent
Attrition                
No          1233    83.88
Yes          237    16.12
Imbalance ratio: 5.20 : 1


## Removing uninformative columns

Drops the constant and identifier columns found above and writes the result to data/processed/cleaned_ibm_dataset.csv. Later steps read that file, so the raw CSV is loaded only once.


In [6]:
drop_cols = list(constant_cols.index) + list(identifier_cols.index)

df_clean = df.drop(columns=drop_cols)

CLEAN_PATH = Path("data/processed/cleaned_ibm_dataset.csv")
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(CLEAN_PATH, index=False)

print("Dropped:", drop_cols)
print("Shape:", df.shape, "->", df_clean.shape)
print("Saved:", CLEAN_PATH)


Dropped: ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
Shape: (1470, 35) -> (1470, 31)
Saved: data\processed\cleaned_ibm_dataset.csv



## Protected attributes

Adds a binary column for each protected attribute, marking the privileged group as 1: male for gender, 40 and over for age, married for marital status. The group sizes and attrition rates are printed so any disparity is visible before a model is trained.


In [7]:
df_clean = pd.read_csv(CLEAN_PATH)

df_clean["gender_priv"] = (df_clean["Gender"] == "Male").astype(int)
df_clean["age_priv"] = (df_clean["Age"] >= 40).astype(int)
df_clean["marital_priv"] = (df_clean["MaritalStatus"] == "Married").astype(int)

PROTECTED = ["gender_priv", "age_priv", "marital_priv"]

rows = []
for col in PROTECTED:
    rate = df_clean.groupby(col)["Attrition"].apply(lambda s: (s == "Yes").mean() * 100)
    rows.append({
        "attribute": col,
        "n_unprivileged": int((df_clean[col] == 0).sum()),
        "n_privileged": int((df_clean[col] == 1).sum()),
        "attrition_pct_unprivileged": round(rate[0], 2),
        "attrition_pct_privileged": round(rate[1], 2),
        "gap": round(rate[0] - rate[1], 2),
    })

df_clean.to_csv(CLEAN_PATH, index=False)

print(pd.DataFrame(rows).to_string(index=False))


   attribute  n_unprivileged  n_privileged  attrition_pct_unprivileged  attrition_pct_privileged   gap
 gender_priv             588           882                       14.80                     17.01 -2.21
    age_priv             948           522                       18.99                     10.92  8.07
marital_priv             797           673                       19.20                     12.48  6.72


Age shows the widest disparity, at 18.99 percent attrition below 40 against 10.92 percent at 40 and over, and marital status is close behind at 19.20 against 12.48. Gender is the narrowest and the only one running the other way, at 14.80 for women against 17.01 for men.

## Splitting and encoding

Splits the data into training and test sets stratified on attrition, then scales the numeric columns and one-hot encodes the categorical ones. The encoder is fitted on the training set only and applied to the test set, so no test information reaches the model.



In [8]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

y = (df_clean["Attrition"] == "Yes").astype(int)
X = df_clean.drop(columns=["Attrition"] + PROTECTED)
protected = df_clean[PROTECTED]

X_train, X_test, y_train, y_test, prot_train, prot_test = train_test_split(
    X, y, protected, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False), categorical_cols),
])

X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.transform(X_test)
feature_names = [n.split("__", 1)[1] for n in preprocessor.get_feature_names_out()]


def assemble(encoded, prot, target):
    out = pd.DataFrame(encoded, columns=feature_names)
    for col in PROTECTED:
        out[col] = prot[col].to_numpy()
    out["Attrition"] = target.to_numpy()
    return out


train_dataset = assemble(X_train_enc, prot_train, y_train)
test_dataset = assemble(X_test_enc, prot_test, y_test)

TRAIN_PATH = Path("data/processed/train_dataset.csv")
TEST_PATH = Path("data/processed/test_dataset.csv")

train_dataset.to_csv(TRAIN_PATH, index=False)
test_dataset.to_csv(TEST_PATH, index=False)
joblib.dump(preprocessor, "data/processed/preprocessor.joblib")

print("Train:", train_dataset.shape, "->", TRAIN_PATH)
print("Test: ", test_dataset.shape, "->", TEST_PATH)
print("Rows overlapping between sets:", len(set(X_train.index) & set(X_test.index)))
print()
print("Attrition / retention split")
print("  overall  %.2f / %.2f" % (y.mean() * 100, 100 - y.mean() * 100))
print("  train    %.2f / %.2f" % (y_train.mean() * 100, 100 - y_train.mean() * 100))
print("  test     %.2f / %.2f" % (y_test.mean() * 100, 100 - y_test.mean() * 100))


Train: (1176, 48) -> data\processed\train_dataset.csv
Test:  (294, 48) -> data\processed\test_dataset.csv
Rows overlapping between sets: 0

Attrition / retention split
  overall  16.12 / 83.88
  train    16.16 / 83.84
  test     15.99 / 84.01
